In [1]:
from sklearn.datasets import fetch_20newsgroups

docs = fetch_20newsgroups(
    subset='train',
    remove=('headers', 'footers', 'quotes')
).data[:3000]

In [2]:
type(docs)

list

In [3]:
docs[10]

'I have a line on a Ducati 900GTS 1978 model with 17k on the clock.  Runs\nvery well, paint is the bronze/brown/orange faded out, leaks a bit of oil\nand pops out of 1st with hard accel.  The shop will fix trans and oil \nleak.  They sold the bike to the 1 and only owner.  They want $3495, and\nI am thinking more like $3K.  Any opinions out there?  Please email me.\nThanks.  It would be a nice stable mate to the Beemer.  Then I\'ll get\na jap bike and call myself Axis Motors!\n\n-- \n-----------------------------------------------------------------------\n"Tuba" (Irwin)      "I honk therefore I am"     CompuTrac-Richardson,Tx\nirwin@cmptrc.lonestar.org    DoD #0826          (R75/6)'

In [4]:
import re

def preprocess(text):
    text = text.lower()                  
    text = re.sub('[^a-z ]', ' ', text)  
    words = text.split()                 
    
    return words





In [5]:
from collections import Counter

def build_vocab(docs):
    counter = Counter()
    
    for doc in docs:
        words = preprocess(doc)   
        counter.update(words)        
    vocab = {}
    i = 0
    
    for word in counter:
        vocab[word] = i
        i += 1
    
    return vocab


In [6]:
def compute_df(docs, vocab):
    df = {word: 0 for word in vocab}
    
    for doc in docs:
        words = preprocess(doc)
        unique_words = set(words)
        
        for word in unique_words:
            if word in df:
                df[word] += 1
    
    return df


In [7]:
import math

def compute_idf(df, N):
    idf = {}
    
    for word, freq in df.items():
        idf[word] = math.log((N + 1) / (freq + 1)) + 1
    
    return idf

In [8]:
def compute_tf(doc, vocab):
    words = preprocess(doc)
    
    tf = {word: 0 for word in vocab}
    
    for word in words:
        if word in tf:
            tf[word] += 1
    
    total_words = len(words)
    
   
    if total_words == 0:
        return tf
    
    for word in tf:
        tf[word] = tf[word] / total_words
    
    return tf

In [9]:
def build_tfidf(docs):
    vocab = build_vocab(docs)
    df = compute_df(docs, vocab)
    idf = compute_idf(df, len(docs))
    
    tfidf_matrix = []
    
    for doc in docs:
        tf = compute_tf(doc, vocab)
        
        vector = []
        for word in vocab:
            vector.append(tf[word] * idf[word])
        
        tfidf_matrix.append(vector)
    
    return tfidf_matrix, vocab   

In [10]:
import numpy as np

def cosine(a, b):
    a = np.array(a)
    b = np.array(b)

    dot = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)

    if norm_a == 0 or norm_b == 0:
        return 0.0   
    return dot / (norm_a * norm_b)

In [11]:
import numpy as np
def search(query, docs, top_k=5):
    tfidf_matrix, vocab = build_tfidf(docs)

    q_vec = np.zeros(len(vocab)) # build query vector
    q_words = preprocess(query)

    for i, w in enumerate(vocab):
        q_vec[i] = q_words.count(w)

    if np.sum(q_vec) > 0:
        q_vec = q_vec / np.sum(q_vec)

    scores = []

    for i, doc_vec in enumerate(tfidf_matrix):
        score = cosine(q_vec, doc_vec)
        scores.append((score, i))

    scores.sort(reverse=True)

    return scores[:top_k]

In [12]:
query = "machine learning neural network"
results = search(query, docs)

for score, idx in results:
    print(score)
    print(docs[idx][:200])
    print("-" * 50)

0.18972886047076928
I just recently bought a 4 MB ram card for my original mac portable 
(backlit) and have since had some bizarre crashes. It happens when I put 
the machine to sleep and wake the machine up. sometimes i
--------------------------------------------------
0.173427261332022
I'm looking for a Singer Featherweight 221 sewing machine (old, black 
sewing machine in black case).

Please contact:
--------------------------------------------------
0.17295693953450444
OK all you experts!
Need answer quick.386 machine ,1.44 floppy ; unable to write to a formated
720 disk.Machine claims that disk is write protected,but it is not.

Note: It 'll read 720's with no prob
--------------------------------------------------
0.14202191007304238

Yeah.  Sounds typical.  Windows makes all sorts of extra demands on hardware,
and therefore your machine can't keep up with things.  Ever notice how when
acessing the floppies in Windows, everything 
-----------------------------------------------